# P8 / 01: OU-optimal entry and exit rules on the S05 residual (both windows)

Protocol: [`001_claude_p8_ou_optimal_thresholds_preregistration.md`](../../discussions/OU_Optimal_Thresholds/001_claude_p8_ou_optimal_thresholds_preregistration.md).

- **`BT` (Bertram):** a symmetric band ±m* per stock and day, maximising net s-units per unit time given the round-trip cost in σ units.
- **`LP` (Lipton & López de Prado):** entry at |s| ≥ 1.25; per-trade take-profit, stop-loss and a 3-half-life limit, chosen from a simulated first-passage table.
- **Incumbent:** S05 with `x10` exits and the earnings filter (P4 `EX`).
- **Common to all:** measured costs, and no entries inside earnings windows.

**Gates** per window and arm (α = 0.05/2):
- **G1** net bootstrap lower bound > 0;
- **G2** net minus incumbent net, paired bootstrap lower bound > 0;
- **G3** net > 0 at 2× cost.

In [ ]:
import json, os, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, p1, p2, p4, p8, s04, s05, s05b
from equity_trading.alpaca import ALPACA_VERSION
from equity_trading.data import vendor_dir, work_dir

SYNTHETIC = os.environ.get("P8_SYNTHETIC") == "1"
REQUIRED = {"p8": "0.1.0", "s05": "0.1.3"}
_have = {"p8": p8.P8_VERSION, "s05": s05.S05_VERSION}
assert _have == REQUIRED, f"stale equity_trading modules {_have}, expected {REQUIRED}: restart the kernel and run again"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})
PC = p8.P8Config()
SCFG = s05.S05Config(close_long=1.0, close_short=-1.0)               # incumbent x10 exits; estimation as S05
YEARS = {"A 2020-2025": list(range(2020, 2026)), "B 2017-2019": [2017, 2018, 2019]}   # 2026 sealed
QUOTE_CACHE = vendor_dir("Alpaca") / "quote_samples" / "15Min_w3s"
PANEL_B = work_dir() / "results" / "p3" / "panel_15Min_2017_2019.npz"
OUT = ROOT / "notebooks" / "OU_Optimal_Thresholds" / ("01_synthetic" if SYNTHETIC else "01")
OUT.mkdir(parents=True, exist_ok=True)
CACHE = work_dir() / "results" / ("p8_synthetic" if SYNTHETIC else "p8") / f"v{p8.P8_VERSION}"
CACHE.mkdir(parents=True, exist_ok=True)
if SYNTHETIC:
    YEARS = {"A 2025": [2025], "B 2024": [2024]}
    PC = replace(PC, sim_paths=500, n_boot=2000)
    from equity_trading.universe import SECTOR_ETF_01, UNIVERSE_01
    ETF_OF = {s: SECTOR_ETF_01[k] for k, v in UNIVERSE_01.items() for s in v}
    EVENTS = None
else:
    constituents = pd.read_csv(ROOT / "src" / "equity_trading" / "reference" / "sp500_constituents_20260913.csv")
    ETF_OF = dict(zip(constituents["Symbol"], constituents["GICS Sector"].map(s05.SECTOR_ETF)))
    EVENTS = p4.edgar_earnings(sorted(ETF_OF), start="2016-01-01", end="2025-12-31", verbose=False)
STOCKS, ETFS = sorted(ETF_OF), sorted(set(ETF_OF.values()))
print(f"synthetic={SYNTHETIC} | versions {_have}")

## 1. The two rules in s units: Bertram's m*(c) and the LP first-passage table

In [ ]:
t0 = time.time()
BT_TAB = p8.bertram_table(PC.c_grid)
lp_path = CACHE / "lp_table.npz"
if lp_path.exists():
    TAB = dict(np.load(lp_path))
else:
    TAB = p8.lp_table(PC); np.savez(lp_path, **TAB)
LP_CH = p8.lp_choice(TAB, PC.c_grid)
print(f"tables in {time.time() - t0:.0f}s")
show_c = [0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
ci = p8.lookup(np.array(PC.c_grid), np.array(show_c))
display(pd.DataFrame({"c (round trip, sigma units)": show_c, "Bertram m*": BT_TAB.to_numpy()[ci].round(3)}).set_index("c (round trip, sigma units)").T)
lp_view = {}
for c_, i in zip(show_c, ci):
    for s0 in (1.25, 2.0, 3.0, 4.0):
        j = int(np.argmin(np.abs(LP_CH["s0"] - s0)))
        lp_view[(c_, s0)] = {"take_profit": LP_CH["tp"][i, j], "stop_beyond_entry": LP_CH["sl"][i, j], "rate": round(float(LP_CH["rate"][i, j]), 3)}
display(pd.DataFrame(lp_view).T.rename_axis(["c", "|s0|"]))

## 2. Per window: data, cost in σ units, incumbent, BT, LP

In [ ]:
RESULTS = {}
for wi, (wname, years) in enumerate(YEARS.items()):
    w = wname.split()[0]
    t0 = time.time()
    if SYNTHETIC:
        panel, _ = s04.synthetic_ohlc_panel(years, seed=3 + wi)
        H = np.full(panel.c.shape, 2e-4, np.float32)
    else:
        panel = (s04.cached_panel(STOCKS + ETFS + ["SPY"], "15Min", years) if w == "A"
                 else p2.panel_cache(STOCKS + ETFS + ["SPY"], "15Min", years, PANEL_B))
        sess = s05b.sample_sessions(years, 4, seed=42)
        samples = pd.concat([pd.read_parquet(QUOTE_CACHE / f"{d}.parquet") for d in sess], ignore_index=True)
        H = s05b.cost_array(panel.index, panel.day_start, panel.symbols, s05b.cost_model(samples))
    have = set(panel.symbols)
    TRADED = [s for s in STOCKS if s in have and ETF_OF[s] in have]
    SC = np.array([panel.symbols.index(s) for s in TRADED]); EC = np.array([panel.symbols.index(ETF_OF[s]) for s in TRADED])
    tag = ("synthetic_" if SYNTHETIC else "") + f"{w}_s05{s05.S05_VERSION}_p{s04.PANEL_VERSION}"
    est_path = CACHE / f"params_{tag}.npz"
    P = dict(np.load(est_path)) if est_path.exists() else s05.estimate_all(panel, SC, EC, SCFG, s05.first_decision_day(SCFG))
    if not est_path.exists():
        np.savez(est_path, **P)
    days = pd.DatetimeIndex(pd.to_datetime(list(panel.days)))
    ev = p4.synthetic_events(days, TRADED, seed=11 + wi) if SYNTHETIC else EVENTS
    EF = p4.event_flags(ev, days, TRADED, 1, 1).to_numpy()
    MASK = ~np.vstack([EF[1:], np.zeros((1, len(TRADED)), bool)])          # P4 EX: no entries on sessions in windows
    CT = p8.cost_sigma_units(P, H, panel, SC, EC)
    v = P["valid"].astype(bool) & np.isfinite(CT)
    mstar = BT_TAB.to_numpy()[p8.lookup(BT_TAB.index.to_numpy(), np.where(v, CT, 1.0))][v]
    dist = pd.DataFrame({"c_sigma": CT[v], "bertram_m_star": mstar}).describe(percentiles=[.1, .25, .5, .75, .9]).T
    print(f"{wname}: panel {panel.c.shape}, traded {len(TRADED)}, data ready in {time.time() - t0:.0f}s", flush=True)
    display(dist.round(4))
    runs = {}
    t0 = time.time()
    for cm in (1.0, 2.0, 0.0):
        runs[("incumbent", cm)] = s05.backtest(panel, P, SC, EC, replace(SCFG, cost_mult=cm), record=(cm == 1.0), half_spread=H, entry_mask=MASK)
        runs[("BT", cm)] = p8.backtest(panel, P, SC, EC, SCFG, "BT", H, CT, PC, bt_tab=BT_TAB, entry_mask=MASK, cost_mult=cm, record=(cm == 1.0))
        runs[("LP", cm)] = p8.backtest(panel, P, SC, EC, SCFG, "LP", H, CT, PC, lp_ch=LP_CH, entry_mask=MASK, cost_mult=cm, record=(cm == 1.0))
    START = runs[("incumbent", 1.0)]["first_day"]
    print(f"{len(runs)} backtests in {(time.time() - t0) / 60:.1f} min", flush=True)
    METRICS = pd.DataFrame({f"{a} {cm:g}x": s04.perf_metrics(r["daily_equity"], SCFG.initial_equity, START) for (a, cm), r in runs.items()}).T
    TS = {}
    for a in ("incumbent", "BT", "LP"):
        tr, r = runs[(a, 1.0)]["trades"], runs[(a, 1.0)]
        TS[a] = {"trades": len(tr), "trades_per_day": len(tr) / max(len(r["daily_equity"][r["daily_equity"].index >= START]), 1),
                 "median_bars": float(tr.bars.median()) if len(tr) else np.nan,
                 "gross_per_trade_bps": 1e4 * tr.gross_ret.mean() if len(tr) else np.nan,
                 "cost_per_trade_bps": 1e4 * r["total_cost"] / tr.notional.sum() if len(tr) else np.nan,
                 **({f"exit_{k}": float(x) for k, x in tr["reason"].value_counts(normalize=True).items()} if len(tr) else {})}
    TS = pd.DataFrame(TS).T
    daily = {a: runs[(a, 1.0)]["daily_equity"].pipe(lambda e: e[e.index >= START].pct_change().dropna()) for a in ("incumbent", "BT", "LP")}
    G = {}
    for a in ("BT", "LP"):
        bn = {b: p1.block_bootstrap(daily[a].to_numpy(), b, PC.n_boot, PC.seed, PC.alpha) for b in PC.blocks}
        bd = {b: p8.paired_bootstrap(daily[a], daily["incumbent"], b, PC.n_boot, PC.seed, PC.alpha) for b in PC.blocks}
        G[a] = {**p8.gates(bn, bd, METRICS.loc[f"{a} 2x"].to_dict()),
                "boot_net_bps": {str(b): [1e4 * x for x in q] for b, q in bn.items()},
                "boot_diff_bps": {str(b): [1e4 * x for x in q] for b, q in bd.items()}}
    YR = pd.DataFrame({a: d.groupby(d.index.year).apply(lambda x: (1 + x).prod() - 1) for a, d in daily.items()})
    display(METRICS.round(4)); display(TS.round(3))
    display(pd.DataFrame({a: {k: v for k, v in g.items() if not k.startswith("boot")} for a, g in G.items()}))
    display((100 * YR).round(2))
    for a in ("incumbent", "BT", "LP"):
        runs[(a, 1.0)]["daily_equity"].to_csv(OUT / f"daily_equity_{w}_{a}.csv", header=["equity"], float_format="%.2f")
    RESULTS[wname] = {"metrics": METRICS.to_dict(orient="index"), "trade_stats": TS.to_dict(orient="index"), "gates": G,
                      "net_by_year": {str(k): x for k, x in YR.to_dict(orient="index").items()}, "cost_and_mstar": dist.to_dict(orient="index")}
    fig, ax = plt.subplots(figsize=(8, 3.4))
    for a, col in zip(("incumbent", "BT", "LP"), ("#444444", "#2a78d6", "#eb6834")):
        e = runs[(a, 1.0)]["daily_equity"]; e = e[e.index >= START]
        ax.plot(e.index, e, color=col, lw=0.9, label=a)
    ax.set_title(f"{wname}: net equity"); ax.legend(frameon=False); plt.tight_layout(); plt.show()
    del panel, H, P, runs

## 3. Verdicts

In [ ]:
V = {}
for a in ("BT", "LP"):
    g = [RESULTS[w]["gates"][a] for w in RESULTS]
    V[a] = {"improvement_verdict": "IMPROVES_ON_S05" if all(x["G2_improvement"] for x in g) else "NO_IMPROVEMENT",
            "edge_verdict": "EDGE_CANDIDATE" if all(x["G1_edge"] and x["G3_double_cost"] for x in g) else "NO_EDGE"}
display(pd.DataFrame(V))
summary = {"study": "P8-01", "synthetic": SYNTHETIC, "years": YEARS, "config": PC.as_dict(), "s05_config": SCFG.as_dict(),
           "results": RESULTS, "verdicts": V,
           "versions": {"package": __version__, "alpaca": ALPACA_VERSION, "s05": s05.S05_VERSION, "p8": p8.P8_VERSION}}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
(OUT / "manifest.json").write_text(json.dumps({"files": sorted(p.name for p in OUT.iterdir()), "cache": str(CACHE)}, indent=2))